# Lab 02: High-Performance Data Processing with Polars & Dask

In this lab, you will explore out-of-core single-node data engineering using:
1. **Polars:** A lightning-fast DataFrame library powered by Apache Arrow and vectorized Rust execution.
2. **Dask:** Parallel, partitioned out-of-core computation.

We will read read-only benchmark data from `/mnt/storage/datasets/` and write transformed, columnar Parquet files to `/mnt/storage/scratch/`.

In [ ]:
import os
import polars as pl
import dask
import dask.dataframe as dd

print(f"⚡ Polars Version: {pl.__version__}")
print(f"📦 Dask Version: {dask.__version__}")

In [ ]:
# 1. Polars: Scan and transform Titanic benchmark dataset
titanic_csv = "/mnt/storage/datasets/kaggle/competitions/titanic/train.csv"
assert os.path.exists(titanic_csv), f"Dataset missing: {titanic_csv}"

# Lazy scan with Polars for out-of-core memory efficiency
q = (
    pl.scan_csv(titanic_csv)
    .filter(pl.col("Age").is_not_null())
    .group_by("Sex")
    .agg([
        pl.col("PassengerId").count().alias("total_passengers"),
        pl.col("Survived").mean().alias("survival_rate"),
        pl.col("Age").mean().alias("avg_age")
    ])
)

df_summary = q.collect()
print("📊 Polars Aggregation Results:")
print(df_summary)
assert df_summary.height > 0, "Empty Polars summary table"

In [ ]:
# 2. Stream IMDb Reviews and Write Partitioned Columnar Parquet to Scratch
imdb_csv = "/mnt/storage/datasets/nlp/imdb/train.csv"
assert os.path.exists(imdb_csv), f"Dataset missing: {imdb_csv}"

output_parquet = "/mnt/storage/scratch/processed_imdb.parquet"
if os.path.exists(output_parquet):
    os.remove(output_parquet)

df_imdb = pl.read_csv(imdb_csv)
df_transformed = df_imdb.with_columns([
    pl.col("review").str.len_chars().alias("review_length"),
    (pl.col("sentiment") == "positive").cast(pl.Int32).alias("is_positive")
])

df_transformed.write_parquet(output_parquet)
assert os.path.exists(output_parquet), "Failed to write Parquet output to scratch"
print(f"💾 Written transformed Parquet to: {output_parquet} ({os.path.getsize(output_parquet)} bytes)")

# Verify with Dask read
ddf = dd.read_parquet(output_parquet)
print(f"🔎 Dask Computed Record Count: {len(ddf)}")
print("🎉 Lab 02 Completed Successfully!")